# Drug Consumption — Préparation des données
Ce notebook charge, nettoie et prépare les features et les cibles.
Il exporte `prepared_data.pkl` utilisé par `nb_models.ipynb`.

## 0. Imports

In [3]:
import pandas as pd
import numpy as np
import pickle
from imblearn.over_sampling import SMOTE
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

## 1. Chargement

In [4]:
COLS = [
    'ID', 'age', 'gender', 'education', 'country', 'ethnicity',
    'nscore', 'escore', 'oscore', 'ascore', 'cscore', 'impulsive', 'ss',
    'alcohol', 'amphet', 'amyl', 'benzos', 'caff', 'cannabis', 'choc',
    'coke', 'crack', 'ecstasy', 'heroin', 'ketamine', 'legalh', 'lsd',
    'meth', 'mushrooms', 'nicotine', 'semer', 'vsa'
]

df = pd.read_csv('../DATA/drug_consumption.csv', header=None, names=COLS)
print(f'Shape brut : {df.shape}')

Shape brut : (1885, 32)


## 2. Nettoyage — suppression des over-claimers (Semeron)

In [5]:
n_before = len(df)
df = df[df['semer'] == 'CL0'].drop(columns=['semer', 'ID']).reset_index(drop=True)
print(f'Lignes supprimées (Semeron) : {n_before - len(df)}')
print(f'Shape final                 : {df.shape}')

Lignes supprimées (Semeron) : 8
Shape final                 : (1877, 30)


## 3. Construction des features (X)

In [6]:
LEGAL  = ['alcohol', 'caff', 'choc', 'nicotine']
RECENT = ['CL3', 'CL4', 'CL5', 'CL6']

# Binarisation des substances légales
for s in LEGAL:
    df[s + '_bin'] = df[s].isin(RECENT).astype(int)

FEATURE_COLS = [
    'age', 'gender', 'education', 'country', 'ethnicity',
    'nscore', 'escore', 'oscore', 'ascore', 'cscore', 'impulsive', 'ss'
] + [s + '_bin' for s in LEGAL]

X = df[FEATURE_COLS].values

print(f'Features : {len(FEATURE_COLS)} colonnes')
print(f'X shape  : {X.shape}')

Features : 16 colonnes
X shape  : (1877, 16)


## 4. Construction des cibles (y) et rééquilibrage par substance

In [7]:
TARGET_SUBSTANCES = [
    'cannabis', 'nicotine', 'benzos', 'ecstasy', 'amphet',
    'coke', 'lsd', 'mushrooms', 'legalh', 'amyl',
    'meth', 'ketamine', 'heroin', 'crack', 'vsa'
]

def get_strategy(y):
    pct_neg = (y == 0).mean() * 100
    if pct_neg < 60:   return 'direct'
    elif pct_neg < 75: return 'smote'
    else:              return 'smote_weights'

smote = SMOTE(random_state=42)
datasets = {}

for substance in TARGET_SUBSTANCES:
    y     = df[substance].isin(RECENT).astype(int).values
    strat = get_strategy(y)

    if strat == 'direct':
        X_r, y_r = X, y
    else:
        X_r, y_r = smote.fit_resample(X, y)

    datasets[substance] = {
        'X':            X_r,
        'y':            y_r,
        'strategy':     strat,
        'class_weight': 'balanced' if strat == 'smote_weights' else None,
        'before':       Counter(y),
        'after':        Counter(y_r),
    }
    print(f'{substance:12s} | {strat:15s} | avant: {Counter(y)} | après: {Counter(y_r)}')

print('\n✅ Datasets prêts')

cannabis     | direct          | avant: Counter({np.int64(1): 991, np.int64(0): 886}) | après: Counter({np.int64(1): 991, np.int64(0): 886})
nicotine     | direct          | avant: Counter({np.int64(1): 1053, np.int64(0): 824}) | après: Counter({np.int64(1): 1053, np.int64(0): 824})
benzos       | smote           | avant: Counter({np.int64(0): 1345, np.int64(1): 532}) | après: Counter({np.int64(0): 1345, np.int64(1): 1345})
ecstasy      | smote           | avant: Counter({np.int64(0): 1364, np.int64(1): 513}) | après: Counter({np.int64(0): 1364, np.int64(1): 1364})
amphet       | smote_weights   | avant: Counter({np.int64(0): 1444, np.int64(1): 433}) | après: Counter({np.int64(0): 1444, np.int64(1): 1444})
coke         | smote_weights   | avant: Counter({np.int64(0): 1463, np.int64(1): 414}) | après: Counter({np.int64(0): 1463, np.int64(1): 1463})
lsd          | smote_weights   | avant: Counter({np.int64(0): 1501, np.int64(1): 376}) | après: Counter({np.int64(0): 1501, np.int64(1): 150

## 5. Export — `prepared_data.pkl`

In [ ]:
export = {
    'datasets':          datasets,
    'X_raw':             X,
    'feature_cols':      FEATURE_COLS,
    'target_substances': TARGET_SUBSTANCES,
    'df':                df,
}

with open('../DATA_MODELE/prepared_data.pkl', 'wb') as f:
    pickle.dump(export, f)

print('✅ prepared_data.pkl exporté')

✅ prepared_data.pkl exporté
